# Set up your own OGC API

In this notebook you'll publish a GeoPackage as a standards-compliant **OGC API – Features** service, using Kartverket's [OGC API starter kit](https://github.com/kartverket/ogc-api-starter) and the same software Kartverket runs in production:

- **backend**: pygeoapi, the API itself (`backend/Dockerfile`)
- **frontend**: Kartverket's web pages on top of the API (`frontend/Dockerfile`)

```
data/demo.gpkg ──(optional: PostGIS)──▶ backend (pygeoapi) ──▶ frontend ──▶ your browser
                                               ▲                    ▲
                                               └── config/pygeoapi-config.yml (this notebook writes it)
```

Run the cells from top to bottom: click ▶ on each cell, or press **Shift+Enter**. **Run All** at the top works too. The cells you're meant to edit are marked ✏️.

> The first time you run a cell, VS Code may ask you to pick a kernel. Choose **Python Environments → .venv** (marked *Recommended*). It's the environment the Codespace set up for this workshop.

In [ ]:
import json
from workshop import *

## 1. Download the software

Downloads Kartverket's backend and frontend images and builds the starter kit on top of them.

In [ ]:
download()

## 2. Your data ✏️

The workshop ships with a placeholder dataset, `data/demo.gpkg`, stored in EPSG:25833 like Kartverket's own data. It has three layers:

| Layer | |
|---|---|
| `fylker` | Norway's 15 counties (polygons) |
| `kommuner` | Norway's 357 municipalities (polygons), each with its county |
| `byer` | 25 towns (points), each with its municipality and county |

The county and municipality boundaries are Kartverket's administrative units, simplified to keep the file small.

**Want to use your own data?** Drag a GeoPackage (`.gpkg`) from your computer into the `data/` folder in the Explorer on the left, change `GPKG` below and re-run the cell. Every layer in the GeoPackage becomes a *collection* in the API. Git ignores your own files in `data/`, so they aren't committed by accident.

**Where should the API read the data from?** Pick a `SOURCE`:

| `SOURCE` | |
|---|---|
| `"gpkg"` | Straight from the GeoPackage file. Simplest, with no database. |
| `"postgis"` | From a PostGIS database. **This is what Kartverket recommends for real services**, and it adds two things: vector tiles (OGC API – Tiles) and downloads (GeoPackage/CSV) in the frontend. You don't need a database of your own: the cell below starts one here in the Codespace and copies your GeoPackage into it. |

In [ ]:
GPKG = "data/demo.gpkg"
SOURCE = "gpkg"   # or "postgis"

layers = prepare_data(GPKG, SOURCE)

## 3. Describe the dataset ✏️

This is the metadata for the service as a whole. It's shown on the landing page and in the API's own description of itself.

Fill in the form, or switch to the **JSON** tab if you'd rather edit the text. Both change the same values, which are saved to `config/part1-metadata.json` as you go, so nothing is lost if you restart. Below, you can see what it becomes in the configuration file (YAML).

In [ ]:
meta = load_metadata("config/part1-metadata.json", layers, SOURCE)
meta.edit_dataset()

## 4. Describe each collection ✏️

One tab per layer in your GeoPackage. Every layer becomes a *collection* in the API.

- `id`: the collection's name in the URL: `/collections/<id>`. Kartverket's [URL standard](https://kartverket.github.io/ogcapi-docs/docs/url-standard-ogc-api) says: lowercase, words separated by hyphens, `æ/ø/å` written as `ae/o/a`, and preferably a plural noun (`kommuner`, not `kommune`). The form checks it as you type.
- `id_field`: the column that uniquely identifies a feature. It becomes the feature's URL: `/collections/<id>/items/<id_field value>`. A meaningful code makes for readable URLs: `/collections/kommuner/items/0301` is Oslo. When there's no such column, use the GeoPackage's own row id, `fid`.
- `title_field` (optional): a human-readable column, used as the feature's name in the frontend.
- Link (optional): a related page, e.g. the official dataset in [Geonorge](https://www.geonorge.no). It's shown in the frontend and the API. More than one? Add them in the JSON tab.

Switched to your own GeoPackage? Its layers get a tab each, filled in with what can be guessed from the data.

In [ ]:
meta.edit_collections()

## 5. Generate the configuration

Turns your metadata into `config/pygeoapi-config.yml`. It's the same file as in the starter kit, and both the backend and the frontend are built with it. Have a look at what came out, and compare it with the [starter kit's example](https://github.com/kartverket/ogc-api-starter/blob/main/config/pygeoapi-config.yml). With `SOURCE = "postgis"` the `providers` block looks just like the starter kit's.

In [ ]:
config = build_config(meta.dataset, meta.collections, layers, SOURCE)
write_config(config)

print(CONFIG_FILE.read_text(encoding="utf-8"))

## 6. Start your API 🚀

In [ ]:
start()
show_links(layers)

Open the **landing page** link above. That's the Kartverket frontend, running on your API.

> The links only work for you while you're signed in to GitHub, because Codespaces ports are private by default. See step 8 to share them.

## 7. How the API works

An OGC API is plain HTTP: you ask for a URL and get an answer back. Your browser, QGIS and a Python script all ask for the same URLs. Here's what happens.

### One address, two answers

The same URL can answer with a web page or with data. The client says which it wants, in the `Accept` header or with `?f=html` / `?f=json` in the URL. That's called *content negotiation*. In this setup the frontend answers when a web page is wanted, and passes every other request on to pygeoapi:

In [ ]:
cid = layers[0]["id"]

# A browser asks for a web page: Kartverket's frontend answers
show_request(f"/collections/{cid}", accept="text/html")

# QGIS or a script asks for data: pygeoapi answers, through the frontend
show_request(f"/collections/{cid}", accept="application/json")

### The API describes itself

Every response has `links` to related resources, and each link's `rel` says what it points to. A client starts at the landing page and follows links, rather than having URLs built into it. That's how QGIS finds your collections from nothing but the address you paste in.

In [ ]:
# The landing page: where every client starts
for link in get_json("/")["links"]:
    print(f"{link['rel']:55} {link.get('title', '')}")

In [ ]:
# What the server says it can do: the standards and parts of standards ("conformance classes") it implements
for uri in sorted(get_json("/conformance")["conformsTo"]):
    print(uri.split("/spec/")[-1])

### Collections and features

The data itself: the collections, and the features in them as GeoJSON.

In [ ]:
# Which collections are there?
for c in get_json("/collections")["collections"]:
    print(c["id"], "-", c["title"])

In [ ]:
# The first three features of the first collection: their ids and attributes
cid = layers[0]["id"]
items = get_json(f"/collections/{cid}/items?limit=3")
for f in items["features"]:
    print(f["id"], f["properties"])

In [ ]:
# One feature, by its id, as GeoJSON
fid = items["features"][0]["id"]
feature = get_json(f"/collections/{cid}/items/{fid}")
print(json.dumps(feature, indent=2, ensure_ascii=False)[:1000])

In [ ]:
# Filter by bounding box (lon/lat), and ask for coordinates in EPSG:25833 instead of WGS84
items = get_json(
    f"/collections/{cid}/items?bbox=4,58,12,62"
    "&crs=http://www.opengis.net/def/crs/EPSG/0/25833"
)
print(len(items["features"]), "features in the bbox")
for f in items["features"][:5]:
    print(f["id"], f["geometry"]["coordinates"])

### Paging: how QGIS gets every feature

An API never sends a whole collection in one go. You get one page (here 20 features by default, at most 1000), plus a `next` link to the following page, and a client keeps following `next` until there isn't one. QGIS does this behind the scenes when it loads a layer.

In [ ]:
# Fetch a collection page by page, by following the "next" links
page = get_json(f"/collections/{cid}/items", {"limit": 10, "skipGeometry": "true"})
print(page.get("numberMatched"), "features in total\n")
for n in range(1, 6):  # at most 5 pages, in case your own data is big
    print(f"Page {n}: {len(page['features'])} features")
    next_link = next((l["href"] for l in page["links"] if l["rel"] == "next"), None)
    if not next_link:
        break
    print(f"  next: {next_link}")
    page = get_json(next_link)

**Try it yourself:** edit the URLs above. Some things to try:

- `?limit=5&offset=10`: paging
- `?<column>=<value>`: filter on an attribute, e.g. `/collections/kommuner/items?fylke=Innlandet`
- `/collections/<collection>/items/<id>`: a single feature, e.g. `/collections/kommuner/items/0301`
- `/collections/<collection>/queryables`: which attributes you can filter on
- `/collections/<collection>/tiles`: the vector tiles (with `SOURCE = "postgis"`). The frontend has a tiles page for each collection too.
- `?f=csv`: the features as CSV, e.g. for Excel

## 8. Open it in other tools (optional)

Your API is private until now: only you, signed in to GitHub, can open it. QGIS and other tools can't sign in, so the cell below makes it **public**. Anyone with the link can then reach your API, so only do this with data you're happy to share. (By hand: **Ports** tab at the bottom of VS Code → right-click port **3000** → **Port Visibility → Public**.)

It then checks that your API can be reached from outside the Codespace, the way QGIS reaches it.

In [ ]:
make_public()
check_qgis(layers)

Use the **landing page** URL from step 6. It's the same address for people, QGIS and scripts:

- **QGIS:** **Layer → Add Layer → Add WFS / OGC API – Features Layer… → New**. Paste the URL, set **Version** to *OGC API – Features*, then **Connect** to list your collections.
- **ArcGIS Pro:** **Insert → Connections → Server → New OGC API Server**, and paste the URL.
- **Excel:** **Data → From Web**, and paste the address of a collection as CSV: `<landing page>/collections/<id>/items?f=csv&limit=1000`. Power BI works the same way (**Get data → Web**).

More in Kartverket's guide: [Bruk i GIS-verktøy](https://kartverket.github.io/ogcapi-docs/docs/bruk).

## 9. Change something

Edit the metadata in step 3 or 4, then re-run **steps 5 and 6**. Changed the GeoPackage or `SOURCE`? Re-run from **step 2**. Reload the page in your browser to see the change.

## Stop

Stops the API (and the database, if you used one). Your files stay as they are; run step 6 again to start it back up.

To stop, remove the `#` and run the cell. It's commented out so that **Run All** doesn't stop your API straight after starting it. If your Codespace goes to sleep, the API comes back by itself when you return.

In [ ]:
# stop()

## Next: part 2

Ready for the setup Kartverket uses for real services? Open **`part2-postgis.ipynb`**. It publishes data straight from a PostGIS database, with vector tiles, downloads and advanced queries.